In [ ]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/processed/training_set.parquet")

print("...Info...")
print(df.info())
print("\n...Tail...")
print(df[["date", "target"]].tail())

print("\n...Samples...")
print(df.sample(n=3))


print("\n...Target vars...")
print(df[["date", "target"]].iloc[0])

print("\n...Looking at points...")
print(df["day"].iloc[0][0])
print(df["day"].iloc[1][0])
print(df["day"].iloc[2][0])

print("...Looking at news...")
print(df["news"].sample(n=5))




...Info...
<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   date    244 non-null    str   
 1   hour    244 non-null    object
 2   day     244 non-null    object
 3   week    244 non-null    object
 4   month   244 non-null    object
 5   news    244 non-null    object
 6   target  244 non-null    int64 
dtypes: int64(1), object(5), str(1)
memory usage: 15.9+ KB
None

...Head...
         date  target
0  2024-11-01       1
1  2024-11-04       1
2  2024-11-05       1
3  2024-11-06       1
4  2024-11-07       0

...Tail...
           date  target
239  2025-10-27       1
240  2025-10-28       1
241  2025-10-29       1
242  2025-10-30       1
243  2025-10-31       1

...Samples...
           date                                               hour  \
65   2025-02-13  [{'open': 8.589958343994188e-05, 'high': 0.000...   
205  2025-09-09  [{'open': -0.0023531400050668247, 'h

In [ ]:
df = pd.read_parquet("../data/processed/training_set_with_semantics.parquet")

print("...Info...")
print(df.info())
print("\n...Head...")

print("\n...Samples...")

print("...Looking at news...")
print(df["news"].iloc[0])
print(df["news"].iloc[1])


In [ ]:
# AI generated validaton

print("--- 1. Target Class Balance ---")
# A healthy, upward-trending stock like AAPL should be roughly 52% to 55% Up days (1s)
# If this is 100% or 0%, the logic is fundamentally broken.
balance = df['target'].value_counts(normalize=True).mul(100).round(2)
print(balance.astype(str) + '%')

print("\n--- 2. Logical Verification (The Eye Test) ---")
# Your pipeline appended today's 8:00 AM to 2:59 PM bars to the end of the 'hour' array.
# Therefore, the very last dictionary in the 'hour' list (index -1) is the 2:00 PM bar!
df['today_2pm_close'] = df['hour'].apply(lambda x: x[-1]['raw_close'])

# Shift the column up by 1 to perfectly align "Tomorrow's 2PM Close" on the same row
df['tomorrow_2pm_close'] = df['today_2pm_close'].shift(-1)

# Calculate what the target *should* be based on absolute math
df['calculated_target'] = (df['tomorrow_2pm_close'] > df['today_2pm_close']).astype(int)

# Print 5 random rows to visually verify the numbers
check_cols = ['date', 'today_2pm_close', 'tomorrow_2pm_close', 'target', 'calculated_target']
print(df[check_cols].sample(n=5))

print("\n--- 3. Mismatch Error Rate ---")
# We exclude the very last row (iloc[:-1]) because 'tomorrow' doesn't exist yet
check_df = df.iloc[:-1]
errors = (check_df['target'] != check_df['calculated_target']).sum()

if errors == 0:
    print(f"Mismatch Count: {errors} ✅ FLawless pipeline!")
else:
    print(f"Mismatch Count: {errors} ❌ Uh oh, we have a leak.")

print("\n--- 4. The Edge Case (Very Last Row) ---")
# Verify your dummy '1' applied correctly to the final day
print(df[['date', 'today_2pm_close', 'target']].tail(1))

--- 1. Target Class Balance ---
target
1    54.92%
0    45.08%
Name: proportion, dtype: str

--- 2. Logical Verification (The Eye Test) ---
           date  today_2pm_close  tomorrow_2pm_close  target  \
195  2025-08-25          227.550             227.970       1   
86   2025-03-17          214.550             212.215       0   
198  2025-08-28          232.870             232.150       0   
202  2025-09-04          238.475             239.955       1   
93   2025-03-26          221.575             224.106       1   

     calculated_target  
195                  1  
86                   0  
198                  0  
202                  1  
93                   1  

--- 3. Mismatch Error Rate ---
Mismatch Count: 0 ✅ FLawless pipeline!

--- 4. The Edge Case (Very Last Row) ---
           date  today_2pm_close  target
243  2025-10-31           271.67       1
